# ATL Tech Arena 2026

This Jupyter notebook explains the basic functionality of the provided code and should help you to get started quickly with this challenge. If there are any questions about the challenge itself or some implementation details, please contact the Huawei team via the challenge platform.

In [ ]:
import numpy as np
import trimesh
from src.dataset import Dataset
import k3d

## Dataset

The [`Dataset`](src/dataset.py) class provides access to the provided data, i.e. 3D mesh of the human subjects as well as their pinna landmarks.

Here we assume that the downloaded 3D meshes live in `data/mesh`, and the downloaded landmarks file are in  `data/landmarks`. The landmark files are named as `<subjectID>_left_ear_landmarks.csv` and `<subjectID>_right_ear_landmarks.csv`, respectively.


In [ ]:
dataset = Dataset(mesh_dir="data/mesh", landmarks_dir="data/landmarks")

print(f"Number of subjects: {len(dataset)}")

The data of each subject can be accessed by index:

In [ ]:
subject_index = 0
mesh, landmarks_left, landmarks_right = dataset[subject_index]

print(f"Shape of left pinna landmarks: {landmarks_left.shape}")
print(f"Shape of right pinna landmarks: {landmarks_right.shape}")
print(f"Shape of vertices: {mesh.vertices.shape}")
print(f"shape of faces: {mesh.faces.shape}")
print(f"Shape of vertex normals: {mesh.vertex_normals.shape}")
print(f"Shape of face normals: {mesh.face_normals.shape}")

Example plots to load the mesh and landmarks for visualization

In [ ]:
plot = k3d.plot()
plot += k3d.mesh(mesh.vertices.tolist(), mesh.faces.tolist(),color=0x808080)
plot += k3d.points(positions=landmarks_left.tolist(),point_size=2,color=0xFF0000) # color red
plot += k3d.points(positions=landmarks_right.tolist(),point_size=2,color=0x00FF00 )# color green
plot.display()

# After the plots, click and hold the left mouse button to rotate the mesh to see it from different angles,
# use mouse wheel to zoom in and out.

And you can iterate over all subjects in the dataset:

In [ ]:
for index, (mesh, landmarks_left, landmarks_right) in enumerate(dataset):
    print(f"Subject {index}: {dataset.get_identifier(index)}, Number of vertices: {mesh.vertices.shape[0]}, Number of faces: {mesh.faces.shape[0]}")


The 85 landmarks of pinna comprised 4 distinct pinna contours as 

(i) Outer helix (25 x 3), 

(ii) Concha outline (30 x 3), 

(iii) Inner helix (20 x 3), and 

(iv) Superior antihelix (10 x 3). 


The following example shows how to access the distinct pinna contours.

In [ ]:
contours = {'outer_helix': (0, 25),
            'concha_outline': (25, 55),
            'inner_helix': (55, 75),
            'superior_antihelix': (75, 85)}

subject_index = 0
mesh, left_ear_landmarks, right_ear_landmarks = dataset[subject_index]

outer_helix =  left_ear_landmarks[slice(*contours['outer_helix'])]
concha_outline =  left_ear_landmarks[slice(*contours['concha_outline'])]
inner_helix =  left_ear_landmarks[slice(*contours['inner_helix'])]
superior_antihelix =  left_ear_landmarks[slice(*contours['superior_antihelix'])]

print("Outer Helix:", outer_helix.shape)
print("Concha Outline:", concha_outline.shape)
print("Inner Helix:", inner_helix.shape)
print("Superior Antihelix:", superior_antihelix.shape)

## Implementation

Each submission needs to implement the `LandmarkExtractor` class. This class will be used for automatic evaluation on a hidden test data set and to update the leaderboard. The cell below loads the actual deterministic v2 pipeline from `src/estimator.py`; it does not use the original random placeholder.

In [ ]:
from pathlib import Path
from src.estimator import LandmarkExtractor

checkpoint_path = Path("checkpoints/final_pipeline.pt")
if not checkpoint_path.is_file():
    raise FileNotFoundError(
        f"Final training is not packaged yet: {checkpoint_path}. "
        "Copy the completed schema-v2 final_pipeline.pt here before running inference."
    )

extractor = LandmarkExtractor(
    checkpoint_path=str(checkpoint_path), seed=42, device="auto"
)
print(f"Loaded final model on {extractor.device}")

## Evaluation

The following cell checks the official output contract and calculates mean Euclidean landmark distance on the included KEMAR example. KEMAR is a packaging smoke sample and was included in final training, so this value must not be reported as hidden-test or cross-validation performance.

In [ ]:
from src.metrics import compute_mean_landmark_distance

subject_distances = []

for index, (mesh, landmarks_left, landmarks_right) in enumerate(dataset):
    
    print(f"Subject {index}: {dataset.get_identifier(index)}")

    gt_left = landmarks_left # ground truth - left pinna landmarks
    gt_right = landmarks_right # ground truth - right pinna landmarks

    # --- Get predictions from your model ---
    pred_left, pred_right = extractor.extract(mesh)
    for ear, prediction in (("left", pred_left), ("right", pred_right)):
        assert prediction.shape == (85, 3), (ear, prediction.shape)
        assert prediction.dtype == np.float32, (ear, prediction.dtype)
        assert np.isfinite(prediction).all(), f"{ear} prediction is not finite"
    
    # --- Compute distances ---
    d_left = compute_mean_landmark_distance(pred_left, gt_left)
    d_right = compute_mean_landmark_distance(pred_right, gt_right)

    # Average left & right for this subject
    d_subject = (d_left + d_right) / 2
    print(f'Average distance: {d_subject:.6f} mm')

    subject_distances.append(d_subject)
print('')

# Final mean over all subjects
mean_distance = np.mean(subject_distances)
print(f'Mean distance across packaged smoke subjects: {mean_distance:.6f} mm')


## Submission

The challenge requires `src/estimator.py`, whose `LandmarkExtractor` class is the evaluation entry point, and permits supporting modules and model files. This model's evaluator ZIP therefore contains `src/__init__.py`, `src/estimator.py`, its nine runtime modules, `checkpoints/final_pipeline.pt`, `requirements.txt`, and `THIRD_PARTY_NOTICES.md`. The checkpoint embeds the locator, landmark model, calibration, transforms, deterministic sampling settings, and surface-projection setting; no dataset, fold JSON, predictions JSON, or external calibration JSON is required at inference.

The notebook, KEMAR demonstration files, images, smoke test, and training code are useful in this working folder but are not required by the automatic evaluator. The final end-of-challenge delivery must additionally satisfy the README requirement for brief algorithm documentation and AI training code.